# GreenVision AI: YOLOv8n-Seg Tree Crown Instance Segmentation

Interactive training and continuation notebook for **GreenVision AI**! This notebook gives you full control over training, resuming, evaluating, and running inference with **YOLOv8n-Seg** for individual tree crown instance segmentation.

---

### 🎯 Hardware Target
- **GPU**: NVIDIA GeForce RTX 3050 Laptop GPU (**4.0 GB VRAM**)
- **System RAM**: 16.0 GB
- **Acceleration**: NVIDIA CUDA 12.4 + Automatic Mixed Precision (AMP)
- **Caching**: `cache=False` strictly enforced to prevent RAM/disk cache spikes.

## 1. System & GPU Environment Verification
Let's confirm that PyTorch detects your NVIDIA RTX 3050 GPU and that CUDA acceleration is enabled.

In [ ]:
import os
import sys
import torch
import ultralytics
from ultralytics import YOLO

print(f"Python Version:      {sys.version.split()[0]}")
print(f"PyTorch Version:     {torch.__version__}")
print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU:          {gpu_name}")
    print(f"Dedicated VRAM:      {vram_gb:.2f} GB")
else:
    print("WARNING: CUDA not detected! Check GPU drivers.")

## 2. Model Checkpoint Selection

- To **continue from `exp016` (Epoch 20)**: Use `C:/GreenVision-AI/runs/yolo_tree_seg/bamforests_baseline_exp016/weights/last.pt`
- To start fresh from base weights: Use `C:/GreenVision-AI/models/yolo/yolov8n-seg.pt`

In [ ]:
EXP016_LAST_CKPT = r"C:/GreenVision-AI/runs/yolo_tree_seg/bamforests_baseline_exp016/weights/last.pt"
BASE_WEIGHTS = r"C:/GreenVision-AI/models/yolo/yolov8n-seg.pt"

active_weights = EXP016_LAST_CKPT if os.path.exists(EXP016_LAST_CKPT) else BASE_WEIGHTS
print(f"Selected Weights: {active_weights}")

model = YOLO(active_weights)
print(f"Model Architecture: {model.model.__class__.__name__}")
print(f"Model Task:         {model.task} (Instance Segmentation)")

## 3. Dataset Configuration (`dataset.yaml`)

In [ ]:
DATASET_YAML = "C:/GreenVision-AI/data/processed/yolo_bamforests/dataset.yaml"

if os.path.exists(DATASET_YAML):
    with open(DATASET_YAML, 'r') as f:
        print(f.read())
else:
    print(f"Dataset YAML not found at: {DATASET_YAML}")

## 4. Key Training Parameters Explained

| Parameter | Value | Why this value? |
| :--- | :--- | :--- |
| **`imgsz`** | `512` | Downscaling from high-res tiles to $512 \times 512$ reduces memory consumption quadratically while preserving crown outlines. |
| **`batch`** | `8` | 8 images per forward pass maximizes throughput within 4GB VRAM. |
| **`amp`** | `True` | Automatic Mixed Precision runs activations in FP16, halving tensor memory without loss of accuracy. |
| **`workers`** | `2` | Number of background data loader processes. On Windows, 2 is optimal to avoid CPU RAM thrashing. |
| **`cache`** | `False` | **STRICTLY FALSE**: Prevents 22+ GB `.npy` disk caching. |

## 5. Dry-Run Configuration Check
Run this check to confirm all files, CUDA device, and parameters before launching training.

In [ ]:
from src.training.train_yolo import verify_and_print_config

config = verify_and_print_config(
    model_weights=active_weights,
    dataset_yaml=DATASET_YAML,
    epochs=40,
    imgsz=512,
    batch=8,
    workers=2,
    device='0',
    amp=True,
    cache=False,
    lr0=0.002,
    project='C:/GreenVision-AI/runs/yolo_tree_seg',
    name='bamforests_baseline_exp016',
    resume_mode=False,
    exist_ok=True
)

## 6. Training Continuation (Epochs 20 to 40)

When ready, set `start_training = True` in the cell below to continue training `exp016`.

In [ ]:
start_training = False

if start_training and os.path.exists(DATASET_YAML):
    model = YOLO(active_weights)
    
    results = model.train(
        data=DATASET_YAML,
        epochs=40,               # Target 40 total epochs
        patience=15,             # Early stopping patience
        batch=8,                 # 8 images per batch
        imgsz=512,               # 512x512 resolution
        device=0,                # NVIDIA RTX 3050
        workers=2,               # Safe Windows DataLoader subprocesses
        amp=True,                # Mixed Precision FP16
        cache=False,             # Strictly False (no .npy disk caches)
        lr0=0.002,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        save=True,
        save_period=5,
        project="C:/GreenVision-AI/runs/yolo_tree_seg",
        name="bamforests_baseline_exp016",
        exist_ok=True,
        # Nadir aerial augmentations
        degrees=180.0,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        plots=True,
        verbose=True
    )
    print("Continuation training completed!")
else:
    print("Training paused. Set start_training = True to execute.")

## 7. Model Evaluation on Holdout Test Split

In [ ]:
TRAINED_WEIGHTS = "C:/GreenVision-AI/runs/yolo_tree_seg/bamforests_baseline_exp016/weights/best.pt"

if os.path.exists(TRAINED_WEIGHTS) and os.path.exists(DATASET_YAML):
    trained_model = YOLO(TRAINED_WEIGHTS)
    metrics = trained_model.val(
        data=DATASET_YAML,
        split='test',
        imgsz=512,
        device=0
    )
    print("="*50)
    print(f"Box mAP@50:       {metrics.box.map50:.4f}")
    print(f"Box mAP@50-95:    {metrics.box.map:.4f}")
    print(f"Mask mAP@50:      {metrics.seg.map50:.4f}")
    print(f"Mask mAP@50-95:   {metrics.seg.map:.4f}")
    print("="*50)
else:
    print(f"Trained weights not found at: {TRAINED_WEIGHTS}")